# Phase 1 + 2: train GPT-1 on Tiny Shakespeare

Runtime → Change runtime type → **T4 GPU** before running anything.

Run the cells top to bottom. To sample again later without retraining,
run cells 1-5 and then the last cell: the checkpoint lives on Drive.

What to expect:
- the line before training: initial loss about 7.69, vs ln(2048) = 7.62
- loss sits near 6.37 early on (the model only knows token frequencies), then drops
- 5000 steps is about 62 passes over the text, so val loss will likely bottom
  out and rise while train keeps falling. The checkpoint is saved at the end,
  which may be past val's best point.

## 1. Clone or update the repo

In [ ]:
import os

REPO = "https://github.com/nabilrakaiza/paperflash.git"

if not os.path.exists("/content/paperflash"):
    !git clone {REPO} /content/paperflash
%cd /content/paperflash
!git pull --ff-only

## 2. Device check

In [ ]:
import torch
print("torch :", torch.__version__)
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — switch the runtime to a GPU")

## 3. Mount Drive

Colab's own disk is wiped when the runtime disconnects, so the checkpoint goes to Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CKPT_DIR = "/content/drive/MyDrive/paperflash/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

## 4. Download the corpus

`data/` is gitignored except for `tokenizer.json`, so the text has to be fetched.
This is the same file used locally, checked by size.

In [ ]:
import urllib.request

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
PATH = "data/input.txt"

if not os.path.exists(PATH):
    urllib.request.urlretrieve(URL, PATH)
assert os.path.getsize(PATH) == 1115394, "corpus differs from the one the tokenizer was trained on"
assert os.path.exists("data/tokenizer.json"), "tokenizer.json missing: it is committed, check the git pull"
print("corpus and tokenizer ready")

## 5. Settings

The config lives in `phase1_baseline/train.py`. Change values here by setting them on
the module before calling `train()` — it reads them at call time.

If you `git pull` new code mid-session, use Runtime → Restart session and run
from the top: Python keeps the old version of a module already imported.

In [ ]:
import phase1_baseline.train as T

T.CHECKPOINT_PATH = os.path.join(CKPT_DIR, "phase1.pt")

print(f"steps {T.MAX_STEPS} | lr {T.LEARNING_RATE} | warmup {int(T.WARMUP_FRAC * T.MAX_STEPS)} steps")
print(f"model N={T.N} d_model={T.D_MODEL} h={T.H} | block {T.BLOCK_SIZE} batch {T.BATCH_SIZE}")
print("checkpoint ->", T.CHECKPOINT_PATH)

## 6. Train

In [ ]:
T.train()

## 7. Samples

Pass `checkpoint_path` explicitly: `sample()`'s default was fixed when it was imported,
so it does not see the Drive path set above.

Same seed, different prompts: if the two continuations are nearly identical,
the model is still ignoring its context.

In [ ]:
from phase2_training.sample import sample

for prompt in ["ROMEO:", "KING HENRY:"]:
    print("=" * 60)
    print(sample(prompt, max_new_tokens=200, seed=0, checkpoint_path=T.CHECKPOINT_PATH))